In [ ]:
import pandas as pd

# 读取Excel文件
df = pd.read_excel('../data/steel_all.xlsx')

# 打印原始列名
print("原始列名:", df.columns.tolist())

# 丢弃"炉号"列（如果存在）
if '炉号' in df.columns:
    X = df.drop(columns=['炉号'])
    print("已丢弃'炉号'列")
else:
    X = df.copy()
    print("数据中不存在'炉号'列")

# 打印处理后的列名
print("处理后列名:", X.columns.tolist())

# 显示处理后数据的前几行
print("\n数据预览:")
print(X.head())

In [ ]:
# 重命名列名为x1, x2, ..., xn，并记录映射关系
n = X.shape[1]  # 获取列数
original_columns = X.columns.tolist()  # 保存原始列名
new_column_names = [f'x{i+1}' for i in range(n)]

# 创建列名映射关系
column_mapping = dict(zip(new_column_names, original_columns))

# 重命名列
X_renamed = X.copy()
X_renamed.columns = new_column_names

# # 保存更换列名后的数据文件
# output_path = 'data/huanlieming.xlsx'
# X_renamed.to_excel(output_path, index=False)
# print(f"重命名后的数据已保存至: {output_path}")


In [ ]:
# 显示列名映射关系
print("Column name mapping (new name -> original name):")
for new_name, old_name in column_mapping.items():
    print(f"{new_name} -> {old_name}")

# 可选：将映射关系保存为Excel文件或CSV文件
mapping_df = pd.DataFrame({
    'New_Column_Name': new_column_names,
    'Original_Column_Name': original_columns
})
# mapping_df.to_excel('../data/column_name_mapping.xlsx', index=False)
# print("\nColumn mapping saved to 'column_name_mapping.xlsx'")

In [ ]:
# 显示修改后的数据框前几行
print("Renamed dataframe first few rows:")
display(X_renamed.head())

# # 将重命名后的数据框保存为xlsx文件
# output_path = '../data.xlsx'
# X_renamed.to_excel(output_path, index=False)
# print(f"\nFile saved successfully to: {output_path}")

# 检查每列的缺失值数量 (使用新列名)
print("\nMissing values count for each column:")
missing_values = X_renamed.isnull().sum()
display(missing_values)

# 计算每列缺失值的百分比 (使用新列名)
print("\nMissing values percentage for each column:")
missing_info = pd.DataFrame({
    'Missing Count': X_renamed.isnull().sum(),
    'Missing Percentage(%)': (X_renamed.isnull().sum() / len(X_renamed)) * 100
})
display(missing_info.sort_values('Missing Percentage(%)', ascending=False))

# 可视化缺失值 (可选)
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.heatmap(X_renamed.isnull(), cmap='viridis', cbar=False, yticklabels=False)
plt.title('Missing Values Visualization')
plt.tight_layout()
plt.show()

填补x15

In [ ]:
# 存储原始的中位数值，用于填充
medians = {}
columns_to_fill = ['x14','x24','x25', 'x32', 'x33', 'x34', 'x35']

for col in columns_to_fill:
    # 计算每列的中位数
    median_value = X_renamed[col].median()
    medians[col] = median_value
    
    # 记录缺失值的数量
    missing_count = X_renamed[col].isnull().sum()
    
    # 使用中位数填充缺失值
    X_renamed[col].fillna(median_value, inplace=True)
    
    # 输出填充信息
    print(f"列 {col}:")
    print(f"  - 缺失值数量: {missing_count}")
    print(f"  - 填充值(中位数): {median_value}")
    print()

# 验证填充后的缺失值数量
print("填充后缺失值检查:")
for col in columns_to_fill:
    print(f"  {col}: {X_renamed[col].isnull().sum()} 缺失值")


# 显示填充后的前几行数据
print("\n填充后的数据预览:")
display(X_renamed[columns_to_fill].head())

In [ ]:
# 检查指定列的缺失率
columns_to_impute = ['x15','x18', 'x26']
missing_rates = {}

print("检查指定列的缺失率:")
for col in columns_to_impute:
    missing_rate = X_renamed[col].isnull().mean()
    missing_rates[col] = missing_rate
    print(f"{col}: {missing_rate:.4f} ({missing_rate*100:.2f}%)")

# 保存插补前的统计数据
print("\n插补前数据统计信息:")
before_stats = pd.DataFrame(index=columns_to_impute)
for col in columns_to_impute:
    # 忽略缺失值计算统计量
    before_stats.loc[col, '平均值'] = X_renamed[col].mean()
    before_stats.loc[col, '中位数'] = X_renamed[col].median()
    before_stats.loc[col, '方差'] = X_renamed[col].var()
    before_stats.loc[col, '标准差'] = X_renamed[col].std()
    
print(before_stats.round(4))

# 1. 基于相关性的智能填补（KNN插补）
# 这种方法考虑了变量之间的关系，比简单的统计填充更准确
from sklearn.impute import KNNImputer

# 检查KNN插补前的缺失值
print("\nKNN插补前缺失值检查:")
for col in columns_to_impute:
    missing_before = X_renamed[col].isnull().sum()
    print(f"{col}: {missing_before} 缺失值")

# 重新进行KNN插补
from sklearn.impute import KNNImputer
import numpy as np

# 仅选择数值型列进行插补
numeric_cols = X_renamed.select_dtypes(include=[np.number]).columns.tolist()
print(f"\n将用于KNN插补的数值列数量: {len(numeric_cols)}")

# 创建插补器
imputer = KNNImputer(n_neighbors=5, weights='distance')

# 创建一个仅包含数值列的副本
X_numeric = X_renamed[numeric_cols].copy()

# 应用KNN插补
X_numeric_imputed = pd.DataFrame(
    imputer.fit_transform(X_numeric),
    columns=numeric_cols,
    index=X_renamed.index
)

# 确认插补后确实没有NA值
has_na_after = X_numeric_imputed.isna().any().any()
print(f"\nKNN插补后，数值列中是否还有NA值: {has_na_after}")

# 将插补后的数值列合并回原始数据框
X_knn_imputed = X_renamed.copy()
for col in numeric_cols:
    X_knn_imputed[col] = X_numeric_imputed[col]

# 详细检查指定列的插补效果
print("\n指定列的KNN插补效果验证:")
for col in columns_to_impute:
    if col in numeric_cols:  # 确保列是数值型的
        before_count = X_renamed[col].isnull().sum()
        after_count = X_knn_imputed[col].isnull().sum()
        print(f"{col}: 插补前 {before_count} 个缺失值，插补后 {after_count} 个缺失值")
        
        # 如果插补后仍有缺失值，展示一些详细信息
        if after_count > 0:
            print(f"  警告: {col} 列在KNN插补后仍有 {after_count} 个缺失值!")
            print(f"  该列的数据类型: {X_renamed[col].dtype}")
            print(f"  数据样例: {X_renamed[col].head()}")
    else:
        print(f"{col}: 不是数值列，无法使用KNN进行插补")

# 保存插补后的统计数据
print("\n插补后数据统计信息:")
after_stats = pd.DataFrame(index=columns_to_impute)
for col in columns_to_impute:
    if col in numeric_cols:
        after_stats.loc[col, '平均值'] = X_knn_imputed[col].mean()
        after_stats.loc[col, '中位数'] = X_knn_imputed[col].median()
        after_stats.loc[col, '方差'] = X_knn_imputed[col].var()
        after_stats.loc[col, '标准差'] = X_knn_imputed[col].std()
print(after_stats.round(4))

# 计算统计量变化
print("\n插补前后统计量变化:")
stats_diff = pd.DataFrame(index=columns_to_impute)
for col in columns_to_impute:
    if col in numeric_cols:
        stats_diff.loc[col, '平均值变化'] = after_stats.loc[col, '平均值'] - before_stats.loc[col, '平均值']
        stats_diff.loc[col, '中位数变化'] = after_stats.loc[col, '中位数'] - before_stats.loc[col, '中位数']
        stats_diff.loc[col, '方差变化'] = after_stats.loc[col, '方差'] - before_stats.loc[col, '方差']
        stats_diff.loc[col, '标准差变化'] = after_stats.loc[col, '标准差'] - before_stats.loc[col, '标准差']
        stats_diff.loc[col, '平均值变化百分比'] = (stats_diff.loc[col, '平均值变化'] / before_stats.loc[col, '平均值']) * 100 if before_stats.loc[col, '平均值'] != 0 else np.nan
        
print(stats_diff.round(4))


In [ ]:
# 检查指定列是否有缺失值
columns_to_check = ['x15','x18', 'x26']

print("检查指定列的缺失值情况:")
print("=" * 50)

# 创建结果数据框
missing_summary = pd.DataFrame(index=columns_to_check)

# 填充缺失值统计
missing_summary['缺失值数量'] = [X_knn_imputed[col].isnull().sum() for col in columns_to_check]
missing_summary['总行数'] = len(X_knn_imputed)
missing_summary['缺失率'] = [X_knn_imputed[col].isnull().mean() for col in columns_to_check]
missing_summary['缺失百分比'] = missing_summary['缺失率'] * 100

# 添加是否有缺失值的标志
missing_summary['是否有缺失'] = [X_knn_imputed[col].isnull().any() for col in columns_to_check]

# 格式化输出
missing_summary['缺失百分比'] = missing_summary['缺失百分比'].map('{:.2f}%'.format)
missing_summary['缺失率'] = missing_summary['缺失率'].map('{:.4f}'.format)

# 显示结果
display(missing_summary)

# 输出详细描述
print("\n详细描述:")
for col in columns_to_check:
    has_missing = X_knn_imputed[col].isnull().any()
    missing_count = X_knn_imputed[col].isnull().sum()
    total_count = len(X_knn_imputed)
    missing_percent = (missing_count / total_count) * 100
    
    if has_missing:
        print(f"- {col}: 存在 {missing_count} 个缺失值，缺失率 {missing_percent:.2f}%")
        
        # 显示部分缺失值的索引位置
        missing_indices = X_knn_imputed[X_knn_imputed[col].isnull()].index.tolist()
        if len(missing_indices) > 5:
            print(f"  缺失值位置索引示例: {missing_indices[:5]} ... (共 {len(missing_indices)} 个)")
        else:
            print(f"  缺失值位置索引: {missing_indices}")
    else:
        print(f"- {col}: 没有缺失值")

# 可视化缺失情况
plt.figure(figsize=(10, 6))
sns.heatmap(X_knn_imputed[columns_to_check].isnull(), cmap='viridis', 
           yticklabels=False, cbar=False)
plt.tight_layout()
plt.show()

# 计算缺失值的相关性 - 检查是否一起缺失
print("\n缺失值相关性分析 (缺失模式是否相关):")
missing_corr = X_knn_imputed[columns_to_check].isnull().corr()
display(missing_corr)

In [ ]:
# 检查指定列的缺失率
columns_to_check = ['x16', 'x17', 'x19', 'x20', 'x23']
missing_rates = {}

print("检查指定列的缺失率:")
for col in columns_to_check:
    missing_rate = X_knn_imputed[col].isnull().mean()
    missing_rates[col] = missing_rate
    print(f"{col}: {missing_rate:.4f} ({missing_rate*100:.2f}%)")

# 丢弃这些高缺失率的列
X_cleaned = X_knn_imputed.drop(columns=columns_to_check)

# 输出删除列后的数据框信息
print(f"\n删除前的列数: {X_knn_imputed.shape[1]}")
print(f"删除后的列数: {X_cleaned.shape[1]}")
print(f"已删除的列: {columns_to_check}")

In [ ]:
# 可视化缺失值 (可选)
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.heatmap(X_cleaned.isnull(), cmap='viridis', cbar=False, yticklabels=False)
plt.title('Missing Values Visualization')
plt.tight_layout()
plt.show()

In [ ]:
# 删除x21和x22列
X_cleaned = X_cleaned.drop(['x21', 'x22'], axis=1)
print(f"已移除 x21 和 x22 列，数据形状变为: {X_cleaned.shape}")

# 保存清理后的数据（使用新列名x1, x2, ...）
cleaned_output_path = '../data/cleaned_data_x_names.xlsx'
# X_cleaned.to_excel(cleaned_output_path, index=False)
# print(f"使用x列名的清理后数据已保存至: {cleaned_output_path}")
X_cleaned

In [ ]:


# 将列名转换回原始列名
# 首先创建反向映射字典（新列名 -> 原列名）
if 'column_mapping' in locals():
    # 创建仅包含X_cleaned中存在列的映射
    current_columns = X_cleaned.columns.tolist()
    valid_mappings = {new_name: old_name for new_name, old_name in column_mapping.items() 
                     if new_name in current_columns}
    
    # 创建恢复原始列名的数据框副本
    X_original_names = X_cleaned.copy()
    X_original_names.columns = [valid_mappings.get(col, col) for col in X_original_names.columns]
    
    # # 保存使用原始列名的数据
    # original_names_path = 'data/cleaned_data_original_names.xlsx'
    # X_original_names.to_excel(original_names_path, index=False)
    # print(f"使用原始列名的清理后数据已保存至: {original_names_path}")
    
    # 显示前几行数据
    print("\n恢复原始列名后的数据预览:")
    display(X_original_names.head())
else:
    print("警告：未找到列名映射信息，无法恢复原始列名")
    print("请确保先执行了创建列名映射的代码")

In [ ]:
# 将X_original_names的列名再次转换为x1, x2, ...格式，但先丢弃x1列
if 'X_original_names' in locals():
    # 获取当前列名（原始列名）
    current_columns = X_original_names.columns.tolist()
    
    # 创建新的x1, x2, ...列名
    new_column_names = [f'x{i+1}' for i in range(len(current_columns))]
    
    # 创建新的映射关系（x列名 -> 原始列名）
    new_column_mapping = dict(zip(new_column_names, current_columns))
    
    # 复制数据框并重命名列
    X_x_names = X_original_names.copy()
    X_x_names.columns = new_column_names
    
    # 保存重命名后的数据
    x_names_path = '../data/new_data_x_names.xlsx'
    X_x_names.to_excel(x_names_path, index=False)
    print(f"使用x列名的处理后数据已保存至: {x_names_path}")
    
    # 将映射关系保存为Excel文件
    mapping_df = pd.DataFrame({
        'X_Column_Name': new_column_names,
        'Original_Column_Name': current_columns
    })
    mapping_path = '../data/new_mapping_complete.xlsx'
    mapping_df.to_excel(mapping_path, index=False)
    print(f"完整的列名映射关系已保存至: {mapping_path}")
    
    # 显示部分映射关系
    print("\n列名映射关系示例（前5个）:")
    for i, (x_col, orig_col) in enumerate(zip(new_column_names, current_columns)):
        if i < 5:  # 只显示前5个
            print(f"{x_col} -> {orig_col}")
    print("...")
else:
    print("错误：X_original_names数据框不存在")

In [ ]:
X_x_names.head()